In [1]:
import torch, os, time, diffusers, gc, threading, subprocess, psutil
from diffusers.utils import load_image
from PIL.PngImagePlugin import PngInfo

from pipeline_flux_fill_with_cfg import FluxFillCFGPipeline
from transformers import T5EncoderModel, CLIPTextModelWithProjection
from diffusers import BitsAndBytesConfig

def memory():
    gi = subprocess.run(['nvidia-smi','--query-gpu=pstate,memory.used,temperature.gpu,utilization.gpu',
                         '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip().split(',')
    vu, ps, util, temp, ram = float(gi[1].strip().replace(" MiB", "")) / 1024, gi[0].strip(), gi[3].strip(), gi[2].strip(), psutil.virtual_memory()
    print(f"   ..." + "\033[96m" + f"VRAM:" + "\033[93m" + f"{vu:.1f}" +  "\033[96m" + f"/24GB " + \
          "\033[93m" + f"{ps} " + "\033[96m" + f"{util} {temp}C | RAM:" + "\033[93m" + f"{ram.used/1024**3:.1f}" + "\033[96m" + f"/64GB")

#utils end                       #utils end                        #utils end                            #utils end


device, dtype = "cuda", torch.bfloat16
base_model_id = "black-forest-labs/FLUX.1-dev"  
onereward_transformer_id = "bytedance-research/OneReward"

prompt="Animal cookie with yellow frosting and pink sprinkles"
negative_prompt="blurry, noisy, low quality, watermark, text"
strength=1.0
num_inference_steps=30
true_cfg=4.0
guidance_scale=30.0 #1.0
memory()
def flush():
    gc.collect()
    torch.cuda.empty_cache()

def save_image(output, prompt):
    filename = f"fluxfill_onereward_quantized_{hex(hash(str(output)))[:5]}.png"
    metadata = PngInfo()
    metadata.add_text("prompt", prompt)
    metadata.add_text("negative_prompt", negative_prompt)
    output.save(filename, pnginfo=metadata)
    os.startfile(filename)


quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=dtype
)
memory()

print("Loading OneReward Transformer in 4-bit...")
transformer = diffusers.FluxTransformer2DModel.from_pretrained(
    onereward_transformer_id,
    subfolder="flux.1-fill-dev-OneReward-transformer",
    torch_dtype=dtype,
    quantization_config=quant_config
)
flush();memory()

print("Loading T5 Text Encoder from base model in 4-bit...")
text_encoder_2 = T5EncoderModel.from_pretrained(base_model_id, subfolder="text_encoder_2", torch_dtype=dtype, quantization_config=quant_config)
flush();memory()



print(f"Loading pipeline components from {base_model_id}...")
pipeline = FluxFillCFGPipeline.from_pretrained(
    base_model_id,
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    torch_dtype=dtype
)
pipeline.to(device)
flush();memory()




image = load_image("theimage.png")
mask = load_image("themask.png")
width, height = image.size


print("Generating image...")
output = pipeline(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=image,
    mask_image=mask,
    guidance_scale=guidance_scale,
    true_cfg=true_cfg,
    strength=strength,
    height=height,
    width=width,
    num_inference_steps=num_inference_steps,
    max_sequence_length=256).images[0]

threading.Thread(target=save_image, args=(output, prompt)).start()
flush();memory()
print("Done!")

   ...VRAM:1.2/24GB P8 0 % 45C | RAM:10.1/64GB
   ...VRAM:1.2/24GB P8 0 % 45C | RAM:10.1/64GB
Loading OneReward Transformer in 4-bit...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

   ...VRAM:7.4/24GB P5 52 % 46C | RAM:10.3/64GB
Loading T5 Text Encoder from base model in 4-bit...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


   ...VRAM:13.6/24GB P3 0 % 47C | RAM:10.4/64GB
Generating image...


  0%|          | 0/30 [00:00<?, ?it/s]

   ...VRAM:13.6/24GB P2 100 % 79C | RAM:11.3/64GB
Done!
